# Post-processed Yahoo CSVs

Loads each CSV via `yahoo_csv_postprocess` and shows shape, index, column names, and a preview.

**Kernel working directory:** project root (`finance_database/`) or this folder (`yahoo_data/`). The next cell fixes `import` paths and **`importlib.reload`s** `yahoo_csv_postprocess` so edits to that file show up without restarting the kernel.

**`yahoo_fundamentals_metrics.csv`:** output is long format (MultiIndex + `metric_source`, `metric_group`, `metric_name`, `metric_value`); the preview uses `reset_index()` so columns match `symbol`, `as_of_timestamp`, `metric_source`, `metric_group`, `metric_name`, `metric_value`, `source_url`, `status`.

**Reuse:** The load cell fills `df_dict` (e.g. `df_dict["df_yahoo_fundamentals_metrics"]`) and binds the same keys as variables (`df_yahoo_fundamentals_metrics`, `df_yahoo_timeseries`, ...).


In [5]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython import get_ipython
from IPython.display import Markdown, display

_here = Path.cwd()
_candidates = [
    _here / "yahoo_data",
    _here,
    _here.parent / "yahoo_data",
]
for p in _candidates:
    if (p / "yahoo_csv_postprocess.py").is_file():
        sys.path.insert(0, str(p.resolve()))
        break

import importlib

import yahoo_csv_postprocess

importlib.reload(yahoo_csv_postprocess)
from yahoo_csv_postprocess import POSTPROCESS_BY_FILENAME

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")

# How many rows to preview per file
HEAD_ROWS = 10

# Optional: restrict to these basenames, or None for all.
# You may omit the ``.csv`` suffix or pass a single string, e.g. ``"yahoo_key_statistics"``.
ONLY_FILES: list[str] | str | None = None

print("POSTPROCESS_BY_FILENAME:", sorted(POSTPROCESS_BY_FILENAME))

POSTPROCESS_BY_FILENAME: ['index_symbol_map.csv', 'yahoo_crawl_log.csv', 'yahoo_fundamentals_metrics.csv', 'yahoo_quote_metrics.csv', 'yahoo_timeseries.csv']


In [6]:
def show_df(name: str, df: pd.DataFrame, head_rows: int = HEAD_ROWS, *, var_name: str) -> None:
    display(Markdown(f"### {name} — variable `{var_name}`"))
    print(f"shape: {df.shape}")
    print(f"index: {df.index.names} ({type(df.index).__name__})")
    cols = list(df.columns)
    if len(cols) > 20:
        print(f"columns ({len(cols)}): {cols[:20]} ... (+{len(cols) - 20} more)")
    else:
        print(f"columns ({len(cols)}): {cols}")
    display(Markdown(f"**head({head_rows})**"))
    preview = df.sample(min(head_rows, len(df)))
    if name == "yahoo_fundamentals_metrics.csv":
        display(
            Markdown(
                "_Long format: index is `(symbol, as_of_timestamp, metric_source, metric_group, metric_name)`. "
                "Preview below uses `reset_index()` so each row shows all fields as columns._"
            )
        )
        flat = preview.reset_index()
        display(flat)
    elif name == "yahoo_key_statistics.csv":
        display(
            Markdown(
                "_Long format: index is `(symbol, timestamp, url, status)`. "
                "Preview below uses `reset_index()` so each row shows all fields as columns._"
            )
        )
        flat = preview.reset_index()
        display(flat)
    else:
        display(preview)
        

In [7]:
_registry = set(POSTPROCESS_BY_FILENAME)
if ONLY_FILES is None:
    names = sorted(_registry)
else:
    # Single string is a common mistake; treat as one filename
    _only = [ONLY_FILES] if isinstance(ONLY_FILES, str) else list(ONLY_FILES)
    wanted: set[str] = set()
    for f in _only:
        base = Path(f).name
        if not base.endswith(".csv"):
            base = f"{base}.csv"
        wanted.add(base)
    unknown = wanted - _registry
    if unknown:
        raise ValueError(
            f"Unknown or unregistered: {sorted(unknown)}. "
            f"Valid names: {sorted(_registry)}"
        )
    names = sorted(wanted)

df_dict: dict[str, pd.DataFrame] = {}
_ipy = get_ipython()
_user_ns = _ipy.user_ns if _ipy is not None else globals()

for csv_name in names:
    var_name = f"df_{Path(csv_name).stem}"
    fn = POSTPROCESS_BY_FILENAME[csv_name]
    try:
        df = fn()
    except FileNotFoundError as e:
        display(Markdown(f"### {csv_name} (`{var_name}` not set)"))
        print(f"skip (file not found): {e}")
        continue
    except Exception as e:
        display(Markdown(f"### {csv_name} (`{var_name}` not set)"))
        print(f"ERROR: {e!r}")
        continue
    df_dict[var_name] = df
    _user_ns[var_name] = df
    show_df(csv_name, df, var_name=var_name)
    print()

print("df_dict keys:", sorted(df_dict))

### index_symbol_map.csv — variable `df_index_symbol_map`

shape: (110, 5)
index: ['symbol'] (Index)
columns (5): ['index_id', 'confidence', 'quoteType', 'matched_name', 'source_url']


**head(10)**

,index_id,confidence,quoteType,matched_name,source_url
symbol,,,,,
BND,BND,direct,etf,Vanguard Total Bond Market ETF,https://finance.yahoo.com/quote/BND
NVDA,NVDA,direct,equity,NVIDIA Corporation,https://finance.yahoo.com/quote/NVDA
MMM,MMM,direct,equity,3M Company,https://finance.yahoo.com/quote/MMM
000019.SS,000019.SS,direct,index,SSE Index (code 000019),https://finance.yahoo.com/quote/0000...
SBUX,SBUX,direct,equity,Starbucks Corporation,https://finance.yahoo.com/quote/SBUX
AED=X,AED=X,direct,currency,AED/USD Exchange Rate,https://finance.yahoo.com/quote/AED=X
^FCHI,^FCHI,direct,index,CAC 40,https://finance.yahoo.com/quote/^FCHI
AAVE-CNY,AAVE-CNY,direct,cryptocurrency,Aave CNY,https://finance.yahoo.com/quote/AAVE...
ZS=F,ZS=F,direct,commodity,Soybean Futures,https://finance.yahoo.com/quote/ZS=F


### yahoo_crawl_log.csv — variable `df_yahoo_crawl_log`

...

### yahoo_timeseries.csv — variable `df_yahoo_timeseries`

shape: (26260, 19)
index: ['symbol', 'date'] (MultiIndex)
columns (19): ['level_open', 'level_high', 'level_low', 'level_close', 'total_return_level', 'ma_50', 'ma_200', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_mid', 'bb_lower', 'stoch_k', 'stoch_d', 'source_url', 'source', 'technical_source_url']


**head(10)**

level_open  level_high  level_low  level_close  total_return_level     ma_50  ...  bb_lower  \
symbol   date                                                                                      ...             
VEA      2007-09-01       47.62       50.31      46.36        50.14                 NaN       NaN  ...       NaN   
RIO      2012-07-01       48.29        49.7      42.64        46.21                 NaN     54.95  ...     39.16   
^HGX     2007-06-01       237.8       238.9      208.4        210.3                 NaN     216.8  ...     191.1   
         2010-09-01          91       98.94         91        97.28                 NaN     137.3  ...     70.35   
MMM      2023-01-01       101.6       108.6      83.75        87.88                 NaN     125.9  ...     86.56   
DE       2006-03-01       38.53       45.99      36.84         42.8                 NaN     23.17  ...     16.63   
JPM      1992-12-01        12.5       14.33      11.83         13.5                 NaN       NaN  ...     4.725   
LIN      2015-04-01       120.6         125      119.8        121.9                 NaN     115.8  ...     119.5   
USDCHF=X 2015-11-01      0.9862       1.033     0.9843         1.03                 NaN     0.932  ...    0.8745   
^DJI     2018-08-01   2.546e+04   2.617e+04  2.497e+04    2.596e+04                 NaN 1.985e+04  ... 1.932e+04   

                     stoch_k  stoch_d                               source_url  source  \
symbol   date                                                                            
VEA      2007-09-01      NaN      NaN  https://query1.finance.yahoo.com/v8/...   yahoo   
RIO      2012-07-01    17.04    15.58  https://query1.finance.yahoo.com/v8/...   yahoo   
^HGX     2007-06-01    33.07    44.95  https://query1.finance.yahoo.com/v8/...   yahoo   
         2010-09-01    22.43    31.95  https://query1.finance.yahoo.com/v8/...   yahoo   
MMM      2023-01-01     4.55    5.755  https://query1.finance.yahoo.com/v8/...   yahoo   
DE       2006-03-01    88.28    88.93  https://query1.finance.yahoo.com/v8/...   yahoo   
JPM      1992-12-01    92.51    84.54  https://query1.finance.yahoo.com/v8/...   yahoo   
LIN      2015-04-01    25.73    34.62  https://query1.finance.yahoo.com/v8/...   yahoo   
USDCHF=X 2015-11-01    99.18    89.65  https://query1.finance.yahoo.com/v8/...   yahoo   
^DJI     2018-08-01    87.79    75.91  https://query1.finance.yahoo.com/v8/...   yahoo   

                                        technical_source_url  
symbol   date                                                 
VEA      2007-09-01  https://query1.finance.yahoo.com/v8/...  
RIO      2012-07-01  https://query1.finance.yahoo.com/v8/...  
^HGX     2007-06-01  https://query1.finance.yahoo.com/v8/...  
         2010-09-01  https://query1.finance.yahoo.com/v8/...  
MMM      2023-01-01  https://query1.finance.yahoo.com/v8/...  
DE       2006-03-01  https://query1.finance.yahoo.com/v8/...  
JPM      1992-12-01  https://query1.finance.yahoo.com/v8/...  
LIN      2015-04-01  https://query1.finance.yahoo.com/v8/...  
USDCHF=X 2015-11-01  https://query1.finance.yahoo.com/v8/...  
^DJI     2018-08-01  https://query1.finance.yahoo.com/v8/...  

[10 rows x 19 columns]


df_dict keys: ['df_index_symbol_map', 'df_yahoo_crawl_log', 'df_yahoo_fundamentals_metrics', 'df_yahoo_quote_metrics', 'df_yahoo_timeseries']
